# imgjit — Colab build/test runner

Runtime must be set to **T4 GPU** (`Runtime → Change runtime type`) before running this notebook.

See `docs/PLAN.md` for which phase this is verifying and `docs/ARCHITECTURE.md` / `docs/PROTOCOL.md`
for design context. Run cells top to bottom; the clone/pull cell is safe to re-run after every
`git push` from your Mac — no need to restart the runtime.

In [ ]:
import os

REPO_URL = "https://github.com/zmx27/Image-Processing-Engine.git"
REPO_DIR = "/content/Image-Processing-Engine"

if os.path.isdir(REPO_DIR):
    %cd {REPO_DIR}
    !git pull
else:
    %cd /content
    !git clone {REPO_URL}
    %cd {REPO_DIR}

## Toolchain / GPU sanity check

Run once per session to confirm you actually got a GPU and to catch CUDA/driver version drift
between sessions before it shows up as a confusing build or runtime failure.

In [ ]:
!nvidia-smi
!nvcc --version
!cmake --version

## Configure + build

`find_package(CUDAToolkit)` in the root `CMakeLists.txt` should report the CUDA backend enabled
here (unlike on a Mac, which always falls back to the CPU-only portable core).

In [ ]:
%cd {REPO_DIR}
!cmake -B build -DCMAKE_BUILD_TYPE=RelWithDebInfo
!cmake --build build -j

## Run tests

In [ ]:
%cd {REPO_DIR}
!ctest --test-dir build --output-on-failure

## Phase-specific runs

Cells below are added incrementally as each phase in `docs/PLAN.md` produces something runnable
(e.g. the Phase 2/3 `imgjit-cli`, the Phase 8 `bench` harness).

### Phase 1 — driver API + JIT spike

`imgjit-spike` inverts a PNG on-GPU twice — once from PTX that `nvcc` built ahead of time
(checkpoint 1a, driver plumbing with JIT out of the picture), then once from PTX that NVRTC
compiled at runtime (checkpoint 1b) — and diffs both against a scalar CPU loop. Inversion is an
integer pointwise op, so the bar is exact equality. It exits non-zero on any mismatch, so the
`ctest` cell above already covers it; run it directly to see the per-checkpoint output and to
inspect the generated PTX.

In [ ]:
%cd {REPO_DIR}
!./build/tools/imgjit-spike tests/testdata/gradient_32x32_rgba.png \
    --out /content/inverted.png --dump-ptx /content/invert_jit.ptx

# The generated PTX is what you read when debugging codegen (Phase 3 adds --dump-source
# for the CUDA side of the same idea).
!head -30 /content/invert_jit.ptx

### Phase 3 — codegen + kernel cache

`imgjit-cli --backend cuda` runs the chain through NVRTC codegen, the kernel cache and the driver
API. `--dump-source` writes the *generated* kernel, which is what you read when debugging — not the
fragments it was assembled from — and `--repeat` makes the cold-compile / warm-hit split visible:
run 1 pays the ~50-200 ms NVRTC compile, runs 2..n hit the cache.

Correctness against the CPU oracle and the compile-counter assertions are `imgjit-gpu-tests`,
already run by the `ctest` cell above as `phase3_gpu_oracle_diff` and `phase3_gpu_kernel_cache`.


In [ ]:
%cd {REPO_DIR}
!./build/tools/imgjit-cli --backend cuda \
    --ops "grayscale,gaussian:1.4,sobel,threshold:0.3" --repeat 5 \
    --dump-source /content/chain.cu --dump-ptx /content/chain.ptx \
    tests/testdata/gradient_32x32_rgba.png /content/phase3_out.png

# The generated kernel. Two stages here: grayscale folds into the gaussian's tap helper
# (a prologue, recomputed per tap), and the threshold folds into the sobel's epilogue.
!cat /content/chain.cu


### Phase 5 — GPU worker behind the server

`imgjit-server --backend cuda` puts the CUDA backend on the worker thread: `cuCtxCreate` once at
startup on that thread, the frame slots allocated as **pinned** host memory that connection threads
`recv()` straight into, and every copy and launch on a **single stream** — deliberately, so a
failure here is unambiguously plumbing rather than async overlap (that is Phase 6).

The correctness gate is `phase5_gpu_server`, already run by the `ctest` cell above.

The two cells below record the Phase 5 baseline in `bench/baseline_phase5.csv`. Only re-run
these if you mean to refresh it — and `rm bench/baseline_phase5.csv` first if so. The Phase 6
comparison does **not** depend on re-running these; it writes its own file (`baseline_phase6.csv`)
with a `--streams 1` row that is the honest current-code equivalent of the Phase 5 pipeline.


In [ ]:
%cd {REPO_DIR}
import subprocess, time, re, os

def start_server(extra_args):
    log = open('/content/server.log', 'w+')
    proc = subprocess.Popen(
        ['./build/tools/imgjit-server', '--port', '0', '--backend', 'cuda',
         '--slots', '8', '--slots-per-conn', '2', '--queue', '16',
         '--max-payload', str(16 * 1024 * 1024), '--output-dir', '/content/phase5_out'] + extra_args,
        stdout=log, stderr=subprocess.STDOUT)
    for _ in range(200):
        log.flush()
        text = open('/content/server.log').read()
        match = re.search(r'127\.0\.0\.1:(\d+)', text)
        if match:
            print(text.strip())
            return proc, int(match.group(1))
        time.sleep(0.1)
    raise RuntimeError('server never reported a port:\n' + open('/content/server.log').read())

def bench(port, label):
    subprocess.run(
        ['./build/bench/imgjit-bench', '--port', str(port), '--connections', '4',
         '--frames', '200', '--width', '1024', '--height', '1024', '--channels', '3',
         '--ops', 'grayscale,gaussian:1.4,sobel,threshold:0.3',
         '--label', label, '--csv', 'bench/baseline_phase5.csv'], check=True)

# Run 1: cold. The first frame on each connection pays the NVRTC compile.
proc, port = start_server([])
bench(port, 'phase5_cuda_cold')
proc.terminate(); proc.wait()
print(open('/content/server.log').read().strip().splitlines()[-1])

In [ ]:
%cd {REPO_DIR}
# Run 2: the same load against a server that compiled the chain at startup. Only
# `cold_first_ms` should move — which is docs/ARCHITECTURE.md's prewarm claim, measured.
proc, port = start_server(['--prewarm', 'grayscale,gaussian:1.4,sobel,threshold:0.3'])
bench(port, 'phase5_cuda_prewarmed')
proc.terminate(); proc.wait()
print(open('/content/server.log').read().strip().splitlines()[-1])

print()
print(open('bench/baseline_phase5.csv').read())

### Phase 6 — async multi-stream pipeline

`--streams` is the A/B axis. `--streams 1` is the Phase 5 pipeline (one frame on the GPU at a
time); `--streams 4` is the Phase 6 one. Everything else about the two runs is identical, so the
comparison is like-for-like under the same harness rather than against a build that no longer
exists.

Read **both** outputs. The CSV answers "measurably faster"; the server's last line answers the
other half of the gate — mean and peak stream occupancy. A `mean in flight` near 1.0 under
`--streams 4` means the pipeline serialized and the throughput came from somewhere else, which is
a failed gate no matter what the FPS says.

The correctness gates are `phase6_gpu_async`, `phase6_gpu_stress` and `phase6_gpu_recovery`,
already run by the `ctest` cell above.


In [ ]:
%cd {REPO_DIR}
# Self-contained: does not depend on the Phase 5 cells having run.
import subprocess, time, re, os

CHAIN = 'grayscale,gaussian:1.4,sobel,threshold:0.3'  # compute-bound: ~11x11 taps/pixel
CHEAP = 'invert'                                       # copy-bound: one pointwise op

def start_server(streams):
    log = open('/content/server.log', 'w+')
    proc = subprocess.Popen(
        ['./build/tools/imgjit-server', '--port', '0', '--backend', 'cuda',
         '--streams', str(streams),
         '--slots', '32', '--slots-per-conn', '8', '--queue', '32',
         '--max-payload', str(16 * 1024 * 1024), '--output-dir', '/content/phase6_out',
         '--prewarm', CHAIN, '--prewarm', CHEAP],
        stdout=log, stderr=subprocess.STDOUT)
    for _ in range(300):
        log.flush()
        m = re.search(r'127\.0\.0\.1:(\d+)', open('/content/server.log').read())
        if m:
            print(open('/content/server.log').read().strip())
            return proc, int(m.group(1))
        time.sleep(0.1)
    raise RuntimeError('server never reported a port:\n' + open('/content/server.log').read())

def bench6(port, label, chain):
    # --no-echo: the echo path is 3 MB back per frame + a client-side memcmp, which on
    # Colab's 2 vCPUs caps throughput below what the GPU can do (with echo on, the cheap
    # `invert` chain barely outruns the heavy one — proof the client, not the GPU, was
    # the bottleneck). Request-only isolates the server + GPU pipeline.
    subprocess.run(
        ['./build/bench/imgjit-bench', '--port', str(port), '--connections', '4', '--window', '8',
         '--frames', '300', '--width', '1024', '--height', '1024', '--channels', '3',
         '--no-echo', '--ops', chain, '--label', label, '--csv', 'bench/baseline_phase6.csv'],
        check=True)

!rm -f bench/baseline_phase6.csv   # imgjit-bench appends; start clean

for streams in (1, 4):
    for chain, tag in ((CHAIN, 'full'), (CHEAP, 'cheap')):
        proc, port = start_server(streams)
        bench6(port, f'phase6_streams{streams}_{tag}', chain)
        proc.terminate(); proc.wait()
        print(open('/content/server.log').read().strip().splitlines()[-1])  # occupancy line
        print()

print(open('bench/baseline_phase6.csv').read())


#### Nsight Systems — the overlap gate

Throughput alone cannot distinguish a pipeline that overlaps from one that got faster for another
reason, so the phase gate also asks for a timeline. The H2D, kernel and D2H rows must be genuinely
interleaved across the four streams, not a single file of segments separated by gaps.


In [ ]:
%cd {REPO_DIR}
# Profiles the server under Nsight Systems, then turns the trace into committable
# artifacts. The .nsys-rep is a tens-of-MB binary and is gitignored.
import subprocess, time, os, glob, shutil, signal, sqlite3, csv
import matplotlib; matplotlib.use('Agg'); import matplotlib.pyplot as plt

CHAIN = 'grayscale,gaussian:1.4,sobel,threshold:0.3'

_cands = [shutil.which('nsys')]
for pat in ('/opt/nvidia/nsight-systems/*/target-linux-x64/nsys',
            '/opt/nvidia/nsight-compute/*/host/target-linux-x64/nsys',
            '/opt/nvidia/nsight-compute/*/target-linux-x64/nsys',
            '/usr/local/cuda*/bin/nsys'):
    _cands += sorted(glob.glob(pat))
NSYS = next((c for c in _cands if c and os.path.exists(c)), None)
assert NSYS, 'no nsys found; the occupancy counter carries the overlap claim without it'
print('using', NSYS); subprocess.run([NSYS, '--version'])

nsys_cmd = [
    NSYS, 'profile', '-o', '/content/phase6', '--trace=cuda', '--force-overwrite', 'true',
    './build/tools/imgjit-server', '--port', '9100', '--backend', 'cuda', '--streams', '4',
    '--slots', '32', '--slots-per-conn', '8', '--queue', '32',
    '--max-payload', str(16 * 1024 * 1024), '--prewarm', CHAIN,
]
log = open('/content/nsys_server.log', 'w+')
proc = subprocess.Popen(nsys_cmd, stdout=log, stderr=subprocess.STDOUT)
time.sleep(20)
subprocess.run(
    ['./build/bench/imgjit-bench', '--port', '9100', '--connections', '4', '--window', '8',
     '--frames', '300', '--width', '1024', '--height', '1024', '--channels', '3',
     '--no-echo', '--ops', CHAIN, '--label', 'nsys_run'], check=True)
proc.send_signal(signal.SIGINT); proc.wait()
print(open('/content/nsys_server.log').read().strip().splitlines()[-1])

# Export to sqlite — version-stable, unlike `nsys stats` report names.
ex = subprocess.run([NSYS, 'export', '--type', 'sqlite', '--force-overwrite', 'true',
                     '-o', '/content/phase6.sqlite', '/content/phase6.nsys-rep'],
                    capture_output=True, text=True)
print(ex.stdout); print(ex.stderr)
ex.check_returncode()

db = sqlite3.connect('/content/phase6.sqlite')
tabs = {r[0] for r in db.execute("select name from sqlite_master where type='table'")}

def cols(t):
    return [r[1] for r in db.execute(f'PRAGMA table_info({t})')]

rows = []  # (start_ns, end_ns, streamId, kind)
if 'CUPTI_ACTIVITY_KIND_KERNEL' in tabs:
    for s, e, st in db.execute('select start, end, streamId from CUPTI_ACTIVITY_KIND_KERNEL'):
        rows.append((s, e, st, 'kernel'))
if 'CUPTI_ACTIVITY_KIND_MEMCPY' in tabs:
    ck = 'copyKind' if 'copyKind' in cols('CUPTI_ACTIVITY_KIND_MEMCPY') else 'CopyKind'
    for s, e, st, k in db.execute(f'select start, end, streamId, {ck} from CUPTI_ACTIVITY_KIND_MEMCPY'):
        rows.append((s, e, st, {1: 'H2D', 2: 'D2H'}.get(k, 'copy')))
assert rows, f'no GPU activity in trace; tables present: {sorted(tabs)}'

t0 = min(r[0] for r in rows)
rows = sorted((s - t0, e - t0, st, k) for s, e, st, k in rows)
streams = sorted({r[2] for r in rows})

# Sweep line: fraction of GPU-busy wall time with >=2 ops running at once.
ev = sorted([(s, 1) for s, _, _, _ in rows] + [(e, -1) for _, e, _, _ in rows])
busy = conc = 0.0; depth = 0
for i in range(len(ev) - 1):
    depth += ev[i][1]; span = ev[i + 1][0] - ev[i][0]
    if depth >= 1: busy += span
    if depth >= 2: conc += span
overlap = 100.0 * conc / busy if busy else 0.0
print(f'\nGPU-busy time with >=2 concurrent ops: {overlap:.1f}%   '
      f'({len(rows)} ops on {len(streams)} streams)')

with open('bench/phase6_gpu_trace.csv', 'w', newline='') as f:
    w = csv.writer(f); w.writerow(['start_ms', 'end_ms', 'dur_ms', 'stream', 'kind'])
    for s, e, st, k in rows:
        w.writerow([f'{s/1e6:.4f}', f'{e/1e6:.4f}', f'{(e-s)/1e6:.4f}', st, k])

# Timeline, first 60 ms — the committable visual. Download /content/phase6.nsys-rep for
# the interactive Nsight Systems view.
col = {'H2D': '#4c78a8', 'kernel': '#e45756', 'D2H': '#54a24b', 'copy': '#888888'}
fig, ax = plt.subplots(figsize=(14, 1 + 0.5 * len(streams)))
for s, e, st, k in rows:
    if s > 6e7:
        continue
    ax.barh(streams.index(st), (e - s) / 1e6, left=s / 1e6,
            color=col.get(k, '#888888'), edgecolor='white', linewidth=0.3)
ax.set_yticks(range(len(streams))); ax.set_yticklabels([f'stream {s}' for s in streams])
ax.set_xlabel('ms')
ax.set_title(f'Phase 6 GPU timeline, 4 streams - {overlap:.0f}% of busy time has >=2 ops')
ax.legend(handles=[plt.Rectangle((0, 0), 1, 1, color=col[c]) for c in ('H2D', 'kernel', 'D2H')],
          labels=['H2D', 'kernel', 'D2H'], loc='upper right')
plt.tight_layout(); plt.savefig('bench/phase6_timeline.png', dpi=120)
print('wrote bench/phase6_timeline.png and bench/phase6_gpu_trace.csv'); plt.show()


### Phase 7 — shared-memory tiling

`--tile` is the A/B axis: `naive` (every stencil tap is a global-memory load) versus a tiled
variant whose block stages its `tile × tile` outputs plus a radius-wide apron in `__shared__`
memory first. Same binary, same harness, only the flag changes.

Read the **server's last line**, not just the CSV. `mean kernel` is the GPU time of the stages
alone, off the GPU's own clock — no copies, no host work, no compile. It is the number the gate
turns on: a 3×3 `sobel` is a sliver of a copy-bound frame, so its `fps` barely moves even when the
kernel gets much faster. Everything runs at `--streams 1`, where that number is a clean per-frame
time; the two `--streams 4` rows at the end are the end-to-end view on top of the Phase 6
pipeline.

The correctness gate is `phase7_gpu_tiling`, already run by the `ctest` cell above.

In [ ]:
%cd {REPO_DIR}
# Self-contained: does not depend on the Phase 5/6 cells having run.
import subprocess, time, re, csv

SHOWCASE = 'grayscale,gaussian:1.4,sobel,threshold:0.3'
CHAINS = [('gaussian', 'gaussian:1.4'), ('sobel', 'sobel'), ('full', SHOWCASE)]
TILES = ['naive', '16', '32']

def start_server7(streams, tile, chain):
    log = open('/content/server.log', 'w+')
    proc = subprocess.Popen(
        ['./build/tools/imgjit-server', '--port', '0', '--backend', 'cuda',
         '--streams', str(streams), '--tile', tile,
         '--slots', '32', '--slots-per-conn', '8', '--queue', '32',
         '--max-payload', str(16 * 1024 * 1024), '--output-dir', '/content/phase7_out',
         '--prewarm', chain],   # warms the variant this server was started with
        stdout=log, stderr=subprocess.STDOUT)
    for _ in range(300):
        m = re.search(r'127\.0\.0\.1:(\d+)', open('/content/server.log').read())
        if m:
            return proc, int(m.group(1))
        time.sleep(0.1)
    raise RuntimeError('server never reported a port:\n' + open('/content/server.log').read())

def run7(streams, tile, tag, chain):
    label = f'phase7_{tag}_{"naive" if tile == "naive" else "tile" + tile}_streams{streams}'
    proc, port = start_server7(streams, tile, chain)
    subprocess.run(
        ['./build/bench/imgjit-bench', '--port', str(port), '--connections', '4', '--window', '8',
         '--frames', '300', '--width', '1024', '--height', '1024', '--channels', '3',
         '--no-echo', '--ops', chain, '--label', label, '--csv', 'bench/baseline_phase7.csv'],
        check=True)
    proc.terminate(); proc.wait()
    last = open('/content/server.log').read().strip().splitlines()[-1]
    print(last)
    return label, float(re.search(r'mean kernel ([\d.]+) ms', last).group(1))

!rm -f bench/baseline_phase7.csv   # imgjit-bench appends; start clean

rows = []  # (tag, tile, streams, label, mean_kernel_ms)
for tag, chain in CHAINS:
    for tile in TILES:
        rows.append((tag, tile, 1) + run7(1, tile, tag, chain))
for tile in ('naive', '16'):   # end-to-end view on top of the Phase 6 pipeline
    rows.append(('full', tile, 4) + run7(4, tile, 'full', SHOWCASE))

with open('bench/phase7_kernel_ms.csv', 'w', newline='') as f:
    w = csv.writer(f); w.writerow(['label', 'chain', 'tile', 'streams', 'mean_kernel_ms'])
    for tag, tile, streams, label, ms in rows:
        w.writerow([label, tag, tile, streams, f'{ms:.4f}'])

# The phase gate: kernel time per chain, tiled vs naive, at --streams 1.
naive = {tag: ms for tag, tile, streams, _, ms in rows if tile == 'naive' and streams == 1}
print()
for tag, tile, streams, _, ms in rows:
    if streams == 1 and tile != 'naive':
        print(f'{tag:8s} tile {tile:>2s}: {naive[tag]:.3f} -> {ms:.3f} ms kernel  '
              f'({naive[tag] / ms:.2f}x)')
print()
print(open('bench/baseline_phase7.csv').read())

### Phase 8 — the benchmark matrix

`docs/PLAN.md` Phase 8 widens the Phase 5 harness over the orthogonal matrix. The cell below runs
**six focused sweeps** rather than one cartesian product: each varies a single axis with the others
pinned, which is what makes a row comparable to the row above it. A full product of six axes would
be hundreds of runs and no clearer about any one of them.

| Sweep | Axis | Pinned | Read |
|---|---|---|---|
| `resolution` | 512² → 4K | streams 4, naive, baked | `fps`, `gb_per_s` |
| `constants` | baked vs parameterized | streams 1, naive | `mean_kernel_ms` |
| `cache` | cold vs warm, four different sigmas | streams 1, naive | `nvrtc_compiles`, `cold_first_ms` |
| `tile_x_streams` | naive/tiled × 1/4 streams | showcase chain | `mean_kernel_ms`, `fps` |
| `fusion` | four one-op chains vs one fused chain | streams 1, naive | `mean_kernel_ms` |
| `chain_length` | 1, 2, 4, 6 ops | streams 1, naive | `mean_kernel_ms`, `fps` |

Two files come out: `bench/baseline_phase8.csv` (what `imgjit-bench` appends, one row per run) and
`bench/phase8_matrix.csv`, which joins those rows to the server-side configuration and instruments
— the axis columns, `mean_kernel_ms` and `nvrtc_compiles` — so the README table can be built from
one file instead of by decoding labels.

**`--streams 1` wherever the gate is kernel time**, for Phase 7's reason: with several streams the
GPU time-slices other frames' kernels into this frame's window, so `mean_kernel_ms` stops being a
clean per-frame number. The `cache` sweep is the one that earns parameterized constants its keep —
four connections asking for four *different* sigmas, cold: baked compiles one kernel per sigma,
parameterized compiles one for all four.

In [ ]:
%cd {REPO_DIR}
# Phase 8 — the benchmark matrix (docs/PLAN.md). Self-contained: does not depend on the
# Phase 5/6/7 cells having run. Expect roughly 5-10 minutes.
import subprocess, time, re, csv

SHOWCASE = 'grayscale,gaussian:1.4,sobel,threshold:0.3'
BASELINE_CSV = 'bench/baseline_phase8.csv'
MATRIX_CSV = 'bench/phase8_matrix.csv'


def start_server(streams, tile, constants, prewarm, slots, slots_per_conn, max_payload):
    log = open('/content/server.log', 'w+')
    args = ['./build/tools/imgjit-server', '--port', '0', '--backend', 'cuda',
            '--streams', str(streams), '--tile', tile, '--constants', constants,
            '--slots', str(slots), '--slots-per-conn', str(slots_per_conn),
            '--queue', '32', '--max-payload', str(max_payload),
            '--output-dir', '/content/phase8_out']
    for chain in prewarm:
        args += ['--prewarm', chain]
    proc = subprocess.Popen(args, stdout=log, stderr=subprocess.STDOUT)
    for _ in range(600):
        match = re.search(r'127\.0\.0\.1:(\d+)', open('/content/server.log').read())
        if match:
            return proc, int(match.group(1))
        time.sleep(0.1)
    raise RuntimeError('server never reported a port:\n' + open('/content/server.log').read())


def run(label, sweep, chains, streams=4, tile='naive', constants='baked', prewarm=None,
        width=1024, height=1024, channels=3, connections=4, window=8, frames=300,
        slots=32, slots_per_conn=8, max_payload=16 * 1024 * 1024):
    """One server + one bench run + one joined row. `chains` is a list: connection i uses
    chain i % len, which is how the cache sweep puts four different sigmas on the wire."""
    prewarm = [] if prewarm is None else prewarm
    proc, port = start_server(streams, tile, constants, prewarm, slots, slots_per_conn,
                              max_payload)
    # --no-echo throughout: the echo is a full frame back per request plus a client-side
    # memcmp, which on Colab's 2 vCPUs caps the loop below what the GPU can do (Phase 6).
    args = ['./build/bench/imgjit-bench', '--port', str(port),
            '--connections', str(connections), '--window', str(window),
            '--frames', str(frames), '--width', str(width), '--height', str(height),
            '--channels', str(channels), '--no-echo',
            '--label', label, '--csv', BASELINE_CSV]
    for chain in chains:
        args += ['--ops', chain]
    subprocess.run(args, check=True, stdout=subprocess.DEVNULL)
    proc.terminate()
    proc.wait()

    log = open('/content/server.log').read()
    stats = re.search(r'mean in flight ([\d.]+).*?mean kernel ([\d.]+) ms, (\d+) NVRTC', log)
    if stats is None:
        raise RuntimeError('server printed no stats line:\n' + log)
    row = [r for r in csv.DictReader(open(BASELINE_CSV)) if r['label'] == label][-1]
    print(f'  {label:38s} {float(row["fps"]):7.1f} fps  p50 {float(row["p50_ms"]):6.2f} ms  '
          f'kernel {float(stats.group(2)):6.3f} ms  {stats.group(3):>2s} compiles')
    return {
        'label': label, 'sweep': sweep, 'chains': ' | '.join(chains),
        'chain_ops': max(len([op for op in c.split(',') if op]) for c in chains),
        'tile': tile, 'streams': streams, 'constants': constants,
        'prewarmed': 'yes' if prewarm else 'no',
        'width': width, 'height': height, 'channels': channels,
        'fps': row['fps'], 'req_mb_per_s': row['req_mb_per_s'],
        'gb_per_s': f'{float(row["req_mb_per_s"]) / 1024.0:.3f}',
        'p50_ms': row['p50_ms'], 'p99_ms': row['p99_ms'],
        'cold_first_ms': row['cold_first_ms'], 'mean_kernel_ms': stats.group(2),
        'mean_in_flight': stats.group(1), 'nvrtc_compiles': stats.group(3),
        'failures': row['failures'],
    }


!rm -f {BASELINE_CSV} {MATRIX_CSV}   # imgjit-bench appends; start clean
rows = []

# Resolution is swept separately (the next cell): a closed-loop harness's fps is
# mechanically tied to latency by Little's Law at fixed concurrency, so the fixed
# 8-frame budget originally used here could not distinguish "GPU-bound" from
# "window-bound", and needs a per-resolution concurrency check the other five sweeps
# do not (they hold concurrency fixed and compare ratios, which cancels the effect).

# 1. Baked vs parameterized, at one stream so mean_kernel_ms is a clean per-frame number.
#    This is the cost side of the axis: dynamic loop bounds NVRTC cannot unroll.
print('\nconstants sweep (baked vs parameterized)')
for constants in ('baked', 'parameterized'):
    for tag, chain in (('gaussian', 'gaussian:1.4'), ('full', SHOWCASE)):
        rows.append(run(f'phase8_{tag}_{constants}', 'constants', [chain], streams=1,
                        constants=constants, prewarm=[chain]))

# 3. The benefit side, and the cache axis at the same time: four connections, four
#    DIFFERENT sigmas. Cold, baked compiles one kernel per sigma and parameterized
#    compiles one for all four; warm, the compiles are out of the measurement entirely.
print('\ncache sweep (four sigmas, cold then warm)')
SIGMAS = ['gaussian:0.8', 'gaussian:1.4', 'gaussian:2.2', 'gaussian:3']
for constants in ('baked', 'parameterized'):
    rows.append(run(f'phase8_sigmas_cold_{constants}', 'cache', SIGMAS, streams=1,
                    constants=constants))
for constants in ('baked', 'parameterized'):
    rows.append(run(f'phase8_sigmas_warm_{constants}', 'cache', SIGMAS, streams=1,
                    constants=constants, prewarm=SIGMAS))

# 4. The Phase 6 and Phase 7 axes crossed, which neither phase ran in full.
print('\ntile x streams sweep')
for tile in ('naive', '16'):
    for streams in (1, 4):
        name = 'naive' if tile == 'naive' else f'tile{tile}'
        rows.append(run(f'phase8_full_{name}_streams{streams}', 'tile_x_streams', [SHOWCASE],
                        streams=streams, tile=tile, prewarm=[SHOWCASE]))

# 5. Fusion. The showcase chain's four ops fuse into two kernels; the unfused baseline is
#    those same four ops as four separate one-op chains, whose kernel times sum. Not a
#    perfect stand-in — four chains are also four round trips — which is why the summary
#    below compares kernel time and not fps.
print('\nfusion sweep (four one-op chains vs one fused chain)')
for op in ('grayscale', 'gaussian:1.4', 'sobel', 'threshold:0.3'):
    rows.append(run(f'phase8_unfused_{op.split(":")[0]}', 'fusion', [op], streams=1,
                    prewarm=[op]))
rows.append(run('phase8_fused_full', 'fusion', [SHOWCASE], streams=1, prewarm=[SHOWCASE]))

# 6. Chain length, one op to six.
print('\nchain length sweep')
for chain in ('invert', 'grayscale,sobel', SHOWCASE,
              'grayscale,brightness:0.1,gaussian:1.4,invert,sobel,threshold:0.3'):
    rows.append(run(f'phase8_len{len(chain.split(","))}', 'chain_length', [chain], streams=1,
                    prewarm=[chain]))

with open(MATRIX_CSV, 'w', newline='') as f:
    writer = csv.DictWriter(f, fieldnames=list(rows[0].keys()))
    writer.writeheader()
    writer.writerows(rows)

# The comparisons the phase is gated on, computed here rather than left to be eyeballed.
at = {r['label']: r for r in rows}
print('\n--- baked vs parameterized, kernel time at 1 stream ---')
for tag in ('gaussian', 'full'):
    baked = float(at[f'phase8_{tag}_baked']['mean_kernel_ms'])
    param = float(at[f'phase8_{tag}_parameterized']['mean_kernel_ms'])
    print(f'  {tag:8s} {baked:.3f} -> {param:.3f} ms  ({param / baked:.2f}x the baked kernel)')

print('--- four sigmas on a cold cache ---')
for constants in ('baked', 'parameterized'):
    cold = at[f'phase8_sigmas_cold_{constants}']
    warm = at[f'phase8_sigmas_warm_{constants}']
    print(f'  {constants:14s} cold {cold["nvrtc_compiles"]:>2s} compiles, '
          f'first frame {float(cold["cold_first_ms"]):6.1f} ms   |   '
          f'warm {warm["nvrtc_compiles"]:>2s} compiles, '
          f'first frame {float(warm["cold_first_ms"]):6.1f} ms')

unfused = sum(float(at[f'phase8_unfused_{op}']['mean_kernel_ms'])
              for op in ('grayscale', 'gaussian', 'sobel', 'threshold'))
fused = float(at['phase8_fused_full']['mean_kernel_ms'])
print(f'--- fusion: 4 separate kernels {unfused:.3f} ms vs 1 fused chain {fused:.3f} ms '
      f'({unfused / fused:.2f}x) ---')

print()
print(open(MATRIX_CSV).read())

### Phase 8 resolution sweep — corrected methodology

The resolution numbers first recorded here used a fixed 8-frame concurrency budget
(`--window 2 --slots 8`) carried over from the Phase 5 baseline, sized for a hardcoded 64 MiB slot
regardless of the frame actually sent. Checking the recorded `fps` against the closed loop's own
theoretical cap (`concurrency / p50`) afterwards showed every row within ~15% of that cap — so the
"Mpx/s peaks at 1024², falls off at 512² and 4K" shape could not be distinguished from "8 frames in
flight was not enough to saturate the pipeline at that size." `imgjit-bench` is a closed loop by
construction (`bench/imgjit-bench.cpp`), so throughput is tied to latency by Little's Law at fixed
concurrency — that is expected and is not itself a bug, but it means a resolution sweep needs
enough concurrency that the *pipeline*, not the client's window, is what limits it.

**The fix has two parts:**

1. **Size the slot to the resolution, not a fixed 64 MiB.** `--max-payload` is set to exactly
   `width * height * channels` for each frame, so pinned host memory scales down for small frames
   instead of always paying the largest size's cost — this is what makes raising concurrency for
   the *small* resolutions cheap.
2. **Raise concurrency until the backend itself reports contention.** `CudaBackend` claims one of
   its 4 stream slots per frame and increments `BackendStats::submit_stalls` every time a `submit()`
   found all of them busy (`src/backend/cuda/cuda_backend.cpp`, `claim_stream()`). That counter is a
   direct signal that requests are queueing for the GPU worker rather than merely waiting on client
   window — unlike an fps comparison across concurrency levels, it needs no second run to interpret.
   Each resolution is run at two concurrency levels (memory-budget permitting); the row where
   `submit_stalls > 0` is the one to trust as the pipeline's real ceiling. A resolution where
   *neither* level shows stalls needs a higher `CONCURRENCY_CEILING` and a rerun — the printed
   summary says so explicitly rather than silently reporting an unsaturated number.

In [ ]:
%cd {REPO_DIR}
# Phase 8 resolution sweep, v2 (docs/PLAN.md; see the markdown cell above for why this
# replaces the resolution rows from the combined Phase 8 cell). Self-contained.
import subprocess, time, re, csv

SHOWCASE = 'grayscale,gaussian:1.4,sobel,threshold:0.3'
RESOLUTIONS = [512, 1024, 2048, 4096]
CONNECTIONS = 4
PINNED_BUDGET_BYTES = 2 * 1024 ** 3   # pinned host memory this sweep may use per run
CONCURRENCY_CEILING = 128             # raise and rerun a resolution if it never saturates
FRAMES_PER_CONN = {512: 400, 1024: 300, 2048: 150, 4096: 80}

RESOLUTION_CSV = 'bench/phase8_resolution.csv'
SUMMARY_CSV = 'bench/phase8_resolution_summary.csv'


def start_server(slots, max_payload, prewarm):
    log = open('/content/server.log', 'w+')
    proc = subprocess.Popen(
        ['./build/tools/imgjit-server', '--port', '0', '--backend', 'cuda',
         '--streams', '4', '--slots', str(slots), '--slots-per-conn', str(slots // CONNECTIONS),
         '--queue', str(max(64, slots)), '--max-payload', str(max_payload),
         '--output-dir', '/content/phase8_out', '--prewarm', prewarm],
        stdout=log, stderr=subprocess.STDOUT)
    for _ in range(600):
        match = re.search(r'127\.0\.0\.1:(\d+)', open('/content/server.log').read())
        if match:
            return proc, int(match.group(1))
        time.sleep(0.1)
    raise RuntimeError('server never reported a port:\n' + open('/content/server.log').read())


def run_at_concurrency(edge, concurrency, frames):
    """concurrency == connections * window == total pinned slots; every connection gets
    an equal share of both. slot size is exactly this resolution's frame, not a fixed
    upper bound, which is what makes testing high concurrency at 512^2 cheap."""
    frame_bytes = edge * edge * 3
    window = concurrency // CONNECTIONS
    label = f'phase8_res{edge}_c{concurrency}'

    proc, port = start_server(concurrency, frame_bytes, SHOWCASE)
    subprocess.run(
        ['./build/bench/imgjit-bench', '--port', str(port), '--connections', str(CONNECTIONS),
         '--window', str(window), '--frames', str(frames), '--width', str(edge),
         '--height', str(edge), '--channels', '3', '--no-echo', '--ops', SHOWCASE,
         '--label', label, '--csv', RESOLUTION_CSV],
        check=True, stdout=subprocess.DEVNULL)
    proc.terminate()
    proc.wait()

    log = open('/content/server.log').read()
    # submit_stalls is the definitive signal (see markdown above): unlike comparing fps
    # across concurrency levels, a single run's stall count needs no second data point.
    stats = re.search(
        r'mean in flight ([\d.]+), peak (\d+), (\d+) submit stalls, \d+ context recreation'
        r'.*?mean kernel ([\d.]+) ms', log)
    if stats is None:
        raise RuntimeError('server printed no stats line:\n' + log)
    row = [r for r in csv.DictReader(open(RESOLUTION_CSV)) if r['label'] == label][-1]
    return {
        'label': label, 'edge': edge, 'concurrency': concurrency, 'window': window,
        'pinned_mib': round(concurrency * frame_bytes / (1024 * 1024), 1),
        'fps': float(row['fps']), 'gb_per_s': float(row['req_mb_per_s']) / 1024.0,
        'p50_ms': float(row['p50_ms']), 'p99_ms': float(row['p99_ms']),
        'mean_kernel_ms': float(stats.group(4)), 'mean_in_flight': float(stats.group(1)),
        'submit_stalls': int(stats.group(3)),
    }


!rm -f {RESOLUTION_CSV} {SUMMARY_CSV}
results = []
print(f'{"res":>7s}  {"conc":>5s}  {"pinned":>9s}  {"fps":>9s}  {"GiB/s":>6s}  {"p50":>9s}  '
      f'{"kernel":>9s}  {"stalls":>7s}  saturated?')

for edge in RESOLUTIONS:
    frame_bytes = edge * edge * 3
    max_by_budget = max(CONNECTIONS, int(PINNED_BUDGET_BYTES // frame_bytes))

    base = min(64, CONCURRENCY_CEILING, max_by_budget)
    base = max(base - base % CONNECTIONS, CONNECTIONS)
    doubled = min(base * 2, CONCURRENCY_CEILING, max_by_budget)
    doubled = max(doubled - doubled % CONNECTIONS, CONNECTIONS)

    levels = sorted({base, doubled})
    if len(levels) == 1:
        print(f'  ({edge}^2: pinned-memory budget only permits one concurrency level, '
              f'{levels[0]} — raise PINNED_BUDGET_BYTES for a second data point here)')

    for concurrency in levels:
        r = run_at_concurrency(edge, concurrency, FRAMES_PER_CONN[edge])
        r['saturated'] = r['submit_stalls'] > 0
        results.append(r)
        note = 'yes' if r['saturated'] else 'NO -- window-bound, raise CONCURRENCY_CEILING'
        print(f'{edge:>6d}²  {concurrency:>5d}  {r["pinned_mib"]:>7.0f}MB  {r["fps"]:>9.1f}  '
              f'{r["gb_per_s"]:>6.3f}  {r["p50_ms"]:>7.2f}ms  {r["mean_kernel_ms"]:>7.3f}ms  '
              f'{r["submit_stalls"]:>7d}  {note}')

with open(SUMMARY_CSV, 'w', newline='') as f:
    writer = csv.DictWriter(f, fieldnames=list(results[0].keys()))
    writer.writeheader()
    writer.writerows(results)

print()
print('The row with submit_stalls > 0 at each resolution is the one to trust as the pipeline\'s')
print('real ceiling -- it means requests were queueing for a free GPU stream, not merely waiting')
print('on the client\'s window. Any resolution with NO saturated row needs a higher')
print('CONCURRENCY_CEILING (memory budget permitting) and a rerun before its number is used.')
print()
print(open(RESOLUTION_CSV).read())